# SoftMeta Chatterbox TTS Server v0.9.1

A100 40GB hotfix for long-form Avatar Talking. The notebook repairs a missing installer automatically and uses safer defaults for 10–30 minute videos.

- Select an **A100 GPU** before running the notebook.
- Colab A100 commonly provides **40GB VRAM**, which is supported by the Ditto profile in this build.
- You may use **Run all** safely.
- Engine: `soft-meta/chatterbox-v2@v0.2.1`
- Server and UI: `soft-meta/Chatterbox-TTS-Server@v0.9.1` with automatic v0.9.0 fallback until the new tag is published.
- Generate Voice: MOSS VoiceGenerator
- Generate Video: Ditto TensorRT with PyTorch fallback

Start with a 10–20 second test. Long videos default to checkpointed two-minute sections on A100 40GB.


In [ ]:
%%bash
set -euo pipefail

apt-get update -qq
apt-get install -y -qq \
  ffmpeg libsndfile1 git git-lfs curl ca-certificates lsof \
  sox libsox-fmt-all build-essential

git lfs install

echo "GPU assigned by Colab:"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true
cd /content
rm -rf /content/bin
mkdir -p /content/bin

MICROMAMBA="/content/bin/micromamba"
MICROMAMBA_VERSION="2.6.2-1"
MICROMAMBA_URL="https://github.com/mamba-org/micromamba-releases/releases/download/${MICROMAMBA_VERSION}/micromamba-linux-64"

curl --fail --location --retry 5 --retry-delay 2 --retry-all-errors \
  --connect-timeout 30 "${MICROMAMBA_URL}" --output "${MICROMAMBA}"
chmod +x "${MICROMAMBA}"
"${MICROMAMBA}" --version

for ENV_NAME in sm311 moss312 avatar310; do
  if "${MICROMAMBA}" env list | awk '{print $1}' | grep -qx "${ENV_NAME}"; then
    "${MICROMAMBA}" env remove -n "${ENV_NAME}" -y || true
  fi
done

"${MICROMAMBA}" create -y -n sm311 -c conda-forge python=3.11 pip
"${MICROMAMBA}" create -y -n moss312 -c conda-forge python=3.12 pip

echo "Main and MOSS environments are ready. The avatar installer creates Python 3.10 next."


In [ ]:
%%bash
set -euo pipefail

MM="/content/bin/micromamba"
cd /content
rm -rf chatterbox-v2 Chatterbox-TTS-Server MOSS-TTS

git clone --branch v0.2.1 --depth 1 https://github.com/soft-meta/chatterbox-v2.git
if ! git clone --branch v0.9.1 --depth 1 https://github.com/soft-meta/Chatterbox-TTS-Server.git; then
  echo "v0.9.1 tag is not published yet. Falling back to v0.9.0 and applying the embedded installer hotfix."
  rm -rf Chatterbox-TTS-Server
  git clone --branch v0.9.0 --depth 1 https://github.com/soft-meta/Chatterbox-TTS-Server.git
fi

# Main Chatterbox environment.
"$MM" run -n sm311 python -m pip install -U pip wheel
"$MM" run -n sm311 python -m pip install "setuptools==80.9.0"
"$MM" run -n sm311 python -m pip install \
  --index-url https://download.pytorch.org/whl/cu124 \
  torch==2.6.0 torchaudio==2.6.0
"$MM" run -n sm311 python -m pip install --no-cache-dir chatterbox-tts==0.1.7
"$MM" run -n sm311 python -m pip install --force-reinstall "setuptools==80.9.0"
"$MM" run -n sm311 python -m pip install --no-deps -e /content/chatterbox-v2
"$MM" run -n sm311 python -m pip install \
  -r /content/Chatterbox-TTS-Server/requirements-colab.txt
"$MM" run -n sm311 python -m pip install --force-reinstall "setuptools==80.9.0"

# Isolated MOSS VoiceGenerator environment.
git clone --depth 1 https://github.com/OpenMOSS/MOSS-TTS.git /content/MOSS-TTS
"$MM" run -n moss312 python -m pip install -U pip wheel setuptools
"$MM" run -n moss312 python -m pip install --no-cache-dir \
  --extra-index-url https://download.pytorch.org/whl/cu128 \
  -e "/content/MOSS-TTS[torch-runtime]"
"$MM" run -n moss312 python -m pip install --no-cache-dir \
  -r /content/Chatterbox-TTS-Server/requirements-voice.txt

echo "Chatterbox server and MOSS VoiceGenerator installation completed."


## Install the A100 Avatar Talking worker

This cell creates an isolated Python 3.10 environment, installs the official Ditto runtime, and downloads its checkpoints. TensorRT is preferred; the system uses the PyTorch checkpoint when TensorRT is unavailable.


In [ ]:
%%bash
set -euo pipefail
export SOFTMETA_SERVER_DIR=/content/Chatterbox-TTS-Server
export SOFTMETA_DITTO_DIR=/content/ditto-talkinghead
export SOFTMETA_AVATAR_ENV=avatar310

INSTALLER="$SOFTMETA_SERVER_DIR/scripts/install_ditto_a100.sh"
if [[ ! -f "$INSTALLER" ]]; then
  echo "The GitHub tag is missing the Ditto installer. Creating the embedded v0.9.1 installer now."
  mkdir -p "$SOFTMETA_SERVER_DIR/scripts"
  cat > "$INSTALLER" <<'SOFTMETA_DITTO_INSTALLER'
#!/usr/bin/env bash
set -euo pipefail

# SoftMeta v0.9.1 A100 40GB Ditto installer.
# Usage:
#   bash scripts/install_ditto_a100.sh /content/bin/micromamba
# Optional environment variables:
#   SOFTMETA_AVATAR_ENV=avatar310
#   SOFTMETA_DITTO_DIR=/content/ditto-talkinghead
#   SOFTMETA_SERVER_DIR=/content/Chatterbox-TTS-Server

MM="${1:-/content/bin/micromamba}"
ENV_NAME="${SOFTMETA_AVATAR_ENV:-avatar310}"
DITTO_DIR="${SOFTMETA_DITTO_DIR:-/content/ditto-talkinghead}"
SERVER_DIR="${SOFTMETA_SERVER_DIR:-/content/Chatterbox-TTS-Server}"

if [[ ! -x "$MM" ]]; then
  echo "micromamba was not found at: $MM" >&2
  exit 2
fi
if [[ ! -f "$SERVER_DIR/requirements-avatar.txt" ]]; then
  echo "SoftMeta server was not found at: $SERVER_DIR" >&2
  exit 2
fi

if "$MM" env list | awk '{print $1}' | grep -qx "$ENV_NAME"; then
  "$MM" env remove -n "$ENV_NAME" -y || true
fi
"$MM" create -y -n "$ENV_NAME" -c conda-forge python=3.10 pip

if command -v nvidia-smi >/dev/null 2>&1; then
  echo "Detected GPU:"
  nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
fi

rm -rf "$DITTO_DIR"
git clone --depth 1 https://github.com/antgroup/ditto-talkinghead.git "$DITTO_DIR"

"$MM" run -n "$ENV_NAME" python -m pip install -U pip wheel setuptools
"$MM" run -n "$ENV_NAME" python -m pip install \
  --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
"$MM" run -n "$ENV_NAME" python -m pip install --no-cache-dir \
  -r "$SERVER_DIR/requirements-avatar.txt"

# TensorRT is preferred on A100. A failed install is not fatal because the
# released Ditto PyTorch backend remains available as a fallback.
set +e
"$MM" run -n "$ENV_NAME" python -m pip install --no-cache-dir \
  --extra-index-url https://pypi.nvidia.com \
  tensorrt==8.6.1
TRT_STATUS=$?
set -e
if [[ $TRT_STATUS -ne 0 ]]; then
  echo "TensorRT 8.6.1 could not be installed. SoftMeta will use Ditto PyTorch."
fi

"$MM" run -n "$ENV_NAME" python - <<PYDOWNLOAD
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="digital-avatar/ditto-talkinghead",
    local_dir=r"$DITTO_DIR/checkpoints",
)
print("Ditto checkpoints downloaded.")
PYDOWNLOAD

AVATAR_PYTHON="$($MM run -n "$ENV_NAME" python -c 'import sys; print(sys.executable)')"
"$MM" run -n "$ENV_NAME" python - <<PYVERIFY
from pathlib import Path
import torch
root = Path(r"$DITTO_DIR")
required = [
    root / "inference.py",
    root / "checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl",
    root / "checkpoints/ditto_pytorch",
]
missing = [str(path) for path in required if not path.exists()]
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
if missing:
    raise SystemExit("Missing Ditto files: " + ", ".join(missing))
print("Ditto PyTorch backend is ready.")
trt_cfg = root / "checkpoints/ditto_cfg/v0.4_hubert_cfg_trt.pkl"
trt_root = root / "checkpoints/ditto_trt_Ampere_Plus"
print("Ditto TensorRT assets ready:", trt_cfg.is_file() and trt_root.is_dir())
PYVERIFY

cat <<SUMMARY

Avatar installation completed.
Set these variables before starting SoftMeta:
  SOFTMETA_AVATAR_PYTHON=$AVATAR_PYTHON
  SOFTMETA_DITTO_DIR=$DITTO_DIR
  SOFTMETA_DITTO_CHECKPOINTS=$DITTO_DIR/checkpoints

Important: review THIRD_PARTY_NOTICES.md before commercial deployment.
SUMMARY
SOFTMETA_DITTO_INSTALLER
  chmod +x "$INSTALLER"
fi

test -f "$INSTALLER"
echo "Using installer: $INSTALLER"
bash "$INSTALLER" /content/bin/micromamba


In [ ]:
%%bash
set -euo pipefail
MM="/content/bin/micromamba"

"$MM" run -n sm311 python - <<'PYMAIN'
import sys
from importlib.metadata import version
import torch
import chatterbox
import perth
from softmeta_chatterbox import SoftMetaChatterboxEngine

print("Main Chatterbox environment")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", version("transformers"))
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable. Change the runtime to an A100 GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("Official Chatterbox package:", chatterbox.__file__)
print("PerTh watermarker callable:", callable(getattr(perth, "PerthImplicitWatermarker", None)))
runtime = SoftMetaChatterboxEngine(device="auto")
print("SoftMeta engine device:", runtime.device)
PYMAIN

"$MM" run -n moss312 python - <<'PYMOSS'
import sys
from importlib.metadata import version
import torch
from transformers import AutoModel, AutoProcessor
from speechbrain.inference.speaker import EncoderClassifier

print("\nMOSS VoiceGenerator environment")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", version("transformers"))
print("SpeechBrain:", version("speechbrain"))
print("CUDA available:", torch.cuda.is_available())
print("AutoModel:", AutoModel.__name__)
print("AutoProcessor:", AutoProcessor.__name__)
print("Speaker checker:", EncoderClassifier.__name__)
PYMOSS

"$MM" run -n avatar310 python - <<'PYAVATAR'
from pathlib import Path
import torch
root = Path('/content/ditto-talkinghead')
print("\nAvatar Talking environment")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print("GPU:", name)
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print("GPU VRAM GB:", round(total_gb, 1))
    if total_gb < 38:
        raise SystemExit("Avatar Talking requires an A100-class GPU with about 40GB VRAM.")
    if total_gb < 48:
        print("A100 40GB profile enabled: checkpointed long-video rendering and reduced working resolution.")
    if 'A100' not in name.upper():
        print("WARNING: v0.9.0 is tuned for A100; another GPU may be slower or incompatible with the bundled TensorRT engines.")
required = [
    root / 'inference.py',
    root / 'checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl',
    root / 'checkpoints/ditto_pytorch',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise SystemExit('Missing Ditto files: ' + ', '.join(missing))
try:
    import tensorrt
    print("TensorRT:", tensorrt.__version__)
except Exception as exc:
    print("TensorRT unavailable; PyTorch fallback will be used:", exc)
print("Avatar Talking verification passed.")
PYAVATAR


In [ ]:
import os
import signal
import socket
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display

PORT = 8004
PROJECT = Path('/content/Chatterbox-TTS-Server')
LOG = Path('/content/softmeta_chatterbox_v091.log')
PID_FILE = Path('/content/softmeta_chatterbox_v091.pid')
MM = '/content/bin/micromamba'

if PID_FILE.exists():
    try:
        os.kill(int(PID_FILE.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass
subprocess.run(f"lsof -t -i:{PORT} | xargs -r kill -9", shell=True, check=False)
LOG.unlink(missing_ok=True)

def env_python(name: str) -> str:
    return subprocess.check_output(
        [MM, 'run', '-n', name, 'python', '-c', 'import sys; print(sys.executable)'],
        text=True,
    ).strip()

env = {
    **os.environ,
    'PYTHONUNBUFFERED': '1',
    'HF_HOME': '/content/hf_home',
    'HF_HUB_CACHE': '/content/hf_home/hub',
    'TRANSFORMERS_CACHE': '/content/hf_home/transformers',
    'SOFTMETA_DEVICE': 'cuda',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True,max_split_size_mb:128',
    'SOFTMETA_AVATAR_GPU_PROFILE': 'a100_40gb',
    'SOFTMETA_MODEL': 'chatterbox',
    'SOFTMETA_VOICE_PYTHON': env_python('moss312'),
    'SOFTMETA_MOSS_MODEL_DIR': '/content/softmeta_models/moss_voice_generator',
    'SOFTMETA_AVATAR_PYTHON': env_python('avatar310'),
    'SOFTMETA_DITTO_DIR': '/content/ditto-talkinghead',
    'SOFTMETA_DITTO_CHECKPOINTS': '/content/ditto-talkinghead/checkpoints',
}
Path(env['HF_HOME']).mkdir(parents=True, exist_ok=True)

log_handle = LOG.open('w', encoding='utf-8', errors='replace')
process = subprocess.Popen(
    [MM, 'run', '-n', 'sm311', 'python', '-u', 'start.py'],
    cwd=PROJECT,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
PID_FILE.write_text(str(process.pid), encoding='utf-8')

def port_open() -> bool:
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=.5):
            return True
    except OSError:
        return False

print('Starting SoftMeta Chatterbox TTS Server v0.9.1...')
for _ in range(360):
    if process.poll() is not None:
        log_handle.flush()
        raise RuntimeError(LOG.read_text(errors='replace')[-20000:])
    if port_open():
        break
    time.sleep(1)
else:
    raise TimeoutError('The server did not open port 8004. Run the log cell below.')

from urllib.request import urlopen
with urlopen(f'http://127.0.0.1:{PORT}/', timeout=15) as response:
    page_status = response.status
    page_preview = response.read(300).decode('utf-8', errors='replace')
if page_status != 200 or '<html' not in page_preview.lower():
    raise RuntimeError(f'Unexpected home-page response. HTTP {page_status}: {page_preview}')

from google.colab.output import eval_js
url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
html = (
    '<p><a href="' + url + '" target="_blank" '
    'style="display:inline-block;padding:13px 19px;background:#5f52e8;color:#fff;'
    'border-radius:8px;text-decoration:none;font-weight:700">'
    'Open SoftMeta Audio and Avatar Studio</a></p>'
    '<p style="font-size:13px;color:#667085">Generate Video appears after the last Audio tab. '
    'A100 40GB detected. Test a short clip before a 10-30 minute checkpointed render.</p>'
)
display(HTML(html))
print('Home page check: HTTP 200 OK')
print('Server PID:', process.pid)
print('Server log:', LOG)


## Recent server log


In [ ]:
from pathlib import Path
log = Path('/content/softmeta_chatterbox_v091.log')
print(log.read_text(errors='replace')[-25000:] if log.exists() else 'No server log yet.')


## Optional: Stop the server

Leave `STOP_SERVER` disabled during normal use. Enable it only when you intentionally want to stop the web server.


In [ ]:
STOP_SERVER = False  # @param {type:"boolean"}

import os
import signal
import subprocess
from pathlib import Path

pid_file = Path('/content/softmeta_chatterbox_v091.pid')
if not STOP_SERVER:
    print('Server remains running. Set STOP_SERVER to True only when you want to stop it.')
else:
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text().strip()), signal.SIGTERM)
        except Exception:
            pass
        pid_file.unlink(missing_ok=True)
    subprocess.run('lsof -t -i:8004 | xargs -r kill -9', shell=True, check=False)
    print('Server stopped.')
